# TrackMe Dashboard Automated API & UI Tests

This notebook validates the operational realism of the TrackMe dashboard by testing:
- Notification API
- Analytics API (all, trends, anomalies)
- Incident assignment and status update
- Geofence creation and notification
- Real-time WebSocket event delivery

All tests use Python's `requests` and `websocket-client` libraries.

In [ ]:
# 1. Import Required Libraries
import requests
import json
import time
from websocket import create_connection

BASE_URL = "http://localhost:3000"
API_HEADERS = {"Content-Type": "application/json"}


## 2. Load and Inspect Data

Test the Notification and Analytics API endpoints for basic connectivity and data structure.

In [ ]:
# Test Notification API (GET)
resp = requests.get(f"{BASE_URL}/api/notifications")
print("Notifications GET status:", resp.status_code)
print("Sample notifications:", resp.json().get("notifications", [])[:2])

# Test Analytics API (GET)
resp = requests.get(f"{BASE_URL}/api/analytics")
print("Analytics GET status:", resp.status_code)
print("Sample analytics:", resp.json().get("analytics", [])[:2])

## 3. Data Preprocessing

Test POST endpoints for notifications and analytics. Validate response and check if new data appears in GET results.

In [ ]:
# Test Notification API (POST)
notif_payload = {"message": "Test notification from notebook", "type": "info"}
resp = requests.post(f"{BASE_URL}/api/notifications", headers=API_HEADERS, data=json.dumps(notif_payload))
print("Notification POST status:", resp.status_code, resp.json())

# Test Analytics API (POST)
analytics_payload = {"type": "test", "message": "Test analytics event from notebook"}
resp = requests.post(f"{BASE_URL}/api/analytics", headers=API_HEADERS, data=json.dumps(analytics_payload))
print("Analytics POST status:", resp.status_code, resp.json())

## 4. Feature Engineering

Test Analytics API for trends and anomalies endpoints.

In [ ]:
# Test Analytics Trends
resp = requests.get(f"{BASE_URL}/api/analytics?mode=trends")
print("Analytics Trends status:", resp.status_code)
print("Trends:", resp.json().get("trends", []))

# Test Analytics Anomalies
resp = requests.get(f"{BASE_URL}/api/analytics?mode=anomalies")
print("Analytics Anomalies status:", resp.status_code)
print("Anomalies:", resp.json().get("anomalies", []))

## 5. Model Selection and Training

Test real-time WebSocket event delivery for notifications and analytics.

In [ ]:
# Test WebSocket notification delivery
try:
    ws = create_connection(f"ws://localhost:3000/api/socketio")
    print("WebSocket connected.")
    # Send a notification event
    notif_payload = {"message": "WebSocket test notification", "type": "info"}
    requests.post(f"{BASE_URL}/api/notifications", headers=API_HEADERS, data=json.dumps(notif_payload))
    # Wait for event
    ws.settimeout(5)
    msg = ws.recv()
    print("WebSocket received:", msg)
    ws.close()
except Exception as e:
    print("WebSocket test failed:", e)

## 6. Model Evaluation

Test incident assignment, status update, and geofence creation with notification validation.

In [ ]:
# Simulate incident assignment/status and geofence creation
incident_id = "INC_TEST_001"
unit_id = "UNIT_TEST_001"

# Assign unit to incident (simulate event)
incident_payload = {
    "id": incident_id,
    "type": "Test Incident",
    "status": "New",
    "location": "6.5,3.3",
    "assignedUnits": [],
    "timeline": [],
    "createdAt": time.strftime("%Y-%m-%dT%H:%M:%S")
}
resp = requests.post(f"{BASE_URL}/api/analytics", headers=API_HEADERS, data=json.dumps({"type": "incident", "message": f"Incident {incident_id} created"}))
print("Incident created (analytics event):", resp.status_code)

# Assign unit
incident_payload["assignedUnits"].append(unit_id)
resp = requests.post(f"{BASE_URL}/api/analytics", headers=API_HEADERS, data=json.dumps({"type": "incident", "message": f"Unit {unit_id} assigned to {incident_id}"}))
print("Unit assigned (analytics event):", resp.status_code)

# Update status
incident_payload["status"] = "Engaged"
resp = requests.post(f"{BASE_URL}/api/analytics", headers=API_HEADERS, data=json.dumps({"type": "incident", "message": f"Incident {incident_id} status updated to Engaged"}))
print("Incident status updated (analytics event):", resp.status_code)

# Create geofence
geofence_payload = {"name": "Test Geofence", "center": [6.5, 3.3], "radius": 500}
resp = requests.post(f"{BASE_URL}/api/geofences", headers=API_HEADERS, data=json.dumps(geofence_payload))
print("Geofence created:", resp.status_code, resp.json())

## 7. Make Predictions

Validate that all events and notifications appear in the dashboard UI and are delivered in real time.